# Notebook 4: Experiment 4 — Multi-Stock Training (80/20)
## A Comparative Analysis of BiLSTM and BiGRU for Stock Price Prediction

**Experiment:** Train on 3 stocks combined, predict on the remaining 1 stock.  
**Train/Test Split:** 80/20 (chronological)  
**Models:** BiLSTM, BiGRU, LSTM, GRU  
**Scaler:** ProportionScaler (÷ 10,501)  
**Metrics:** MSE, RMSE, MAE, MAPE, R² Score  

**Combinations:**
- Train TLKM+BBCA+ASII → Predict UNVR
- Train TLKM+BBCA+UNVR → Predict ASII
- Train TLKM+ASII+UNVR → Predict BBCA
- Train BBCA+ASII+UNVR → Predict TLKM


In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from stock_prediction_utils import *

import plotly.graph_objects as go
from plotly.subplots import make_subplots

set_seed()
set_ieee_style()

DATA_DIR = 'dataset'

TRAIN_RATIO = 0.8
RATIO_LABEL = '80_20'
EXP_LABEL = f'Exp4_{RATIO_LABEL}'

os.makedirs(f'figures/{EXP_LABEL}', exist_ok=True)
os.makedirs(f'models/{EXP_LABEL}', exist_ok=True)
os.makedirs('results', exist_ok=True)

print(f"Experiment 4 - Multi-Stock Training (80/20)")


stock_prediction_utils.py loaded successfully!
  ProportionScaler max value: 10501.0
  Lookback: 60, Epochs: 100, Batch size: 64
  Architecture: 2 layers, 64 units, dropout=0.2
  Stocks: ['TLKM', 'BBCA', 'ASII', 'UNVR']
  Models: ['BiLSTM', 'BiGRU', 'LSTM', 'GRU']

GPU SETUP - CUDA Available
Number of GPUs detected: 1
  GPU 0: /physical_device:GPU:0

Memory growth enabled (dynamic allocation)
TensorFlow configured to use GPU


Device Configuration:
  GPUs available: 1
  CPUs available: 1
  TensorFlow will use GPU for computations
Experiment 4 - Multi-Stock Training (80/20)


In [2]:
# Load all daily data
print("Loading daily data...")
daily_data = load_all_daily_data(DATA_DIR)
print("\nAll daily data loaded!")


Loading daily data...
  TLKM: 5243 records, Date range: 2004-09-28 to 2025-12-31
  BBCA: 5244 records, Date range: 2004-09-28 to 2025-12-31
  ASII: 5244 records, Date range: 2004-09-28 to 2025-12-31
  UNVR: 5245 records, Date range: 2004-09-28 to 2025-12-31

All daily data loaded!


## Define Training Combinations

In [3]:
# ============================================================
# MULTI-STOCK COMBINATIONS
# ============================================================
# Each entry: (training_stocks, target_stock)
combinations = []
for target in STOCKS:
    train_stocks = [s for s in STOCKS if s != target]
    combinations.append((train_stocks, target))
    print(f"  Train: {', '.join(train_stocks)} -> Predict: {target}")


  Train: BBCA, ASII, UNVR -> Predict: TLKM
  Train: TLKM, ASII, UNVR -> Predict: BBCA
  Train: TLKM, BBCA, UNVR -> Predict: ASII
  Train: TLKM, BBCA, ASII -> Predict: UNVR


## Run All Multi-Stock Experiments

In [4]:
# ============================================================
# EXPERIMENT 4: Multi-stock training
# ============================================================
all_results = []
all_predictions = {}

for train_stocks, target_stock in combinations:
    train_label = '+'.join(train_stocks)
    pair_key = (train_label, target_stock)
    
    print(f"\n{'#'*60}")
    print(f"# TRAIN: {train_label} -> TARGET: {target_stock}")
    print(f"{'#'*60}")
    
    # Prepare multi-stock data
    train_dfs = [daily_data[s] for s in train_stocks]
    test_df = daily_data[target_stock]
    
    X_train, y_train, X_test, y_test, test_dates = prepare_multi_stock_data(
        train_dfs, test_df,
        train_ratio=TRAIN_RATIO, lookback=LOOKBACK
    )
    print(f"  X_train (combined): {X_train.shape}, X_test: {X_test.shape}")
    
    all_predictions[pair_key] = {}
    
    for model_type in MODEL_TYPES:
        exp_name = f'{EXP_LABEL}_train_{train_label}_target_{target_stock}'
        
        y_true_inv, y_pred_inv, metrics, history = train_and_evaluate(
            model_type=model_type,
            X_train=X_train, y_train=y_train,
            X_test=X_test, y_test=y_test,
            experiment_name=exp_name,
            save_dir=f'models/{EXP_LABEL}',
            epochs=EPOCHS, batch_size=BATCH_SIZE
        )
        
        result = {
            'Train_Stocks': train_label,
            'Target_Stock': target_stock,
            'Model': model_type,
            **metrics
        }
        all_results.append(result)
        all_predictions[pair_key][model_type] = (y_true_inv, y_pred_inv, test_dates)
        
        plot_actual_vs_predicted(
            test_dates, y_true_inv, y_pred_inv,
            model_type, f'Train_{train_label}_Target_{target_stock}',
            EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
        )

print("\n\nAll Experiment 4 (80/20) training complete!")



############################################################
# TRAIN: BBCA+ASII+UNVR -> TARGET: TLKM
############################################################
  X_train (combined): (12406, 60, 1), X_test: (1049, 60, 1)

Training BiLSTM for: Exp4_80_20_train_BBCA+ASII+UNVR_target_TLKM
  Train samples: 12406, Test samples: 1049
Epoch 1/100
175/175 [==============================] - ETA: 0s - loss: 0.0029
Epoch 1: val_loss improved from inf to 0.00030, saving model to models/Exp4_80_20\Exp4_80_20_train_BBCA+ASII+UNVR_target_TLKM_BiLSTM_best.keras
175/175 [==============================] - 21s 55ms/step - loss: 0.0029 - val_loss: 3.0467e-04
Epoch 2/100
175/175 [==============================] - ETA: 0s - loss: 6.4852e-04
Epoch 2: val_loss did not improve from 0.00030
175/175 [==============================] - 7s 42ms/step - loss: 6.4852e-04 - val_loss: 3.1887e-04
Epoch 3/100
175/175 [==============================] - ETA: 0s - loss: 5.6550e-04
Epoch 3: val_loss improved from 0.00030 to

## Results Summary

In [5]:
# ============================================================
# RESULTS TABLE
# ============================================================
results_df = pd.DataFrame(all_results)
print_results_table(results_df, f"Experiment 4 - Multi-Stock Training (80/20)")

results_df.to_csv(f'results/{EXP_LABEL}_results.csv', index=False)
print(f"Results saved to results/{EXP_LABEL}_results.csv")



  Experiment 4 - Multi-Stock Training (80/20)
  Train_Stocks Target_Stock  Model        MSE     RMSE      MAE  MAPE (%)       R2  Training_Time_s  Epochs_Run
BBCA+ASII+UNVR         TLKM BiLSTM  4214.8083  64.9216  48.7329    1.5994 0.975328            573.9         100
BBCA+ASII+UNVR         TLKM  BiGRU  3785.9062  61.5297  46.0793    1.5081 0.977839            527.9         100
BBCA+ASII+UNVR         TLKM   LSTM  6048.8863  77.7746  62.6780    2.0066 0.964593            324.1         100
BBCA+ASII+UNVR         TLKM    GRU 10774.4113 103.7999  90.8518    2.8870 0.936931            306.1         100
TLKM+ASII+UNVR         BBCA BiLSTM 91950.6058 303.2336 250.8468    2.9069 0.914927            674.7         100
TLKM+ASII+UNVR         BBCA  BiGRU 92272.2478 303.7635 275.7658    3.3198 0.914629            745.2         100
TLKM+ASII+UNVR         BBCA   LSTM 16873.2983 129.8973  96.6097    1.1622 0.984389            409.2         100
TLKM+ASII+UNVR         BBCA    GRU 14226.4884 119.2748  8

## Visualizations

In [ ]:
# ============================================================
# INTERACTIVE RESULTS VISUALIZATIONS
# ============================================================

print("Generating interactive results visualizations...\n")

# 1. Interactive Metrics Comparison
print("1. Generating Metrics Comparison Chart...")
fig1, html1 = create_interactive_metrics_comparison(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}',
    metrics=['RMSE', 'MAE', 'R2']
)
print(f"   ✓ Saved: {html1}")
fig1.show()

print()

# 2. Model Radar Chart
print("2. Generating Model Radar Chart...")
fig2, html2 = create_interactive_model_radar_chart(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
)
print(f"   ✓ Saved: {html2}")
fig2.show()

print()

# 3. Interactive Dashboard with Subplots
print("3. Generating Results Dashboard...")
fig3 = go.Figure()

# RMSE by model
rmse_by_model = results_df.groupby('Model')['RMSE'].mean().sort_values()
fig3.add_trace(go.Bar(
    x=rmse_by_model.index,
    y=rmse_by_model.values,
    name='RMSE',
    marker_color='#0072B2',
    text=np.round(rmse_by_model.values, 4),
    textposition='outside',
    hovertemplate='Model: %{x}<br>Avg RMSE: %{y:.4f}<extra></extra>'
))

fig3.update_layout(
    title=f"<b>{EXP_LABEL} - Average Metrics by Model</b>",
    xaxis_title="Model",
    yaxis_title="RMSE",
    height=600,
    template='plotly_white',
    font=dict(size=12),
    showlegend=False
)

html3 = f'figures/{EXP_LABEL}/{EXP_LABEL}_metrics_dashboard.html'
fig3.write_html(html3)
print(f"   ✓ Saved: {html3}")
fig3.show()

print("\n✓ All interactive results visualizations generated successfully!")

## Interactive Results Visualizations

In [6]:
# ============================================================
# COMPARISON PLOTS
# ============================================================
for (train_label, target_stock), preds_dict in all_predictions.items():
    y_true = preds_dict[MODEL_TYPES[0]][0]
    dates = preds_dict[MODEL_TYPES[0]][2]
    preds = {mt: preds_dict[mt][1] for mt in MODEL_TYPES if mt in preds_dict}
    
    plot_all_models_comparison(
        dates, y_true, preds,
        f'Train_{train_label}_Target_{target_stock}',
        EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
    )

# Metrics bar chart
for metric in ['RMSE', 'MAE', 'MAPE (%)', 'R2']:
    plot_metrics_comparison_bar(
        results_df, metric, EXP_LABEL,
        group_col='Target_Stock', save_dir=f'figures/{EXP_LABEL}'
    )

print("All visualizations saved!")


  Figure saved: figures/Exp4_80_20/Exp4_80_20_Train_BBCA+ASII+UNVR_Target_TLKM_all_models.png
  Figure saved: figures/Exp4_80_20/Exp4_80_20_Train_TLKM+ASII+UNVR_Target_BBCA_all_models.png
  Figure saved: figures/Exp4_80_20/Exp4_80_20_Train_TLKM+BBCA+UNVR_Target_ASII_all_models.png
  Figure saved: figures/Exp4_80_20/Exp4_80_20_Train_TLKM+BBCA+ASII_Target_UNVR_all_models.png
  Figure saved: figures/Exp4_80_20/Exp4_80_20_RMSE_comparison.png
  Figure saved: figures/Exp4_80_20/Exp4_80_20_MAE_comparison.png
  Figure saved: figures/Exp4_80_20/Exp4_80_20_MAPE_pct_comparison.png
  Figure saved: figures/Exp4_80_20/Exp4_80_20_R2_comparison.png
All visualizations saved!


In [7]:
# ============================================================
# SUMMARY
# ============================================================
print("\n" + "="*70)
print("  BEST MODEL PER TARGET STOCK (by RMSE)")
print("="*70)
for target in STOCKS:
    target_data = results_df[results_df['Target_Stock'] == target]
    if target_data.empty:
        continue
    best_idx = target_data['RMSE'].idxmin()
    best = target_data.loc[best_idx]
    print(f"  Target {target}: Trained on {best['Train_Stocks']} + {best['Model']} "
          f"(RMSE={best['RMSE']:.4f}, R²={best['R2']:.6f})")



  BEST MODEL PER TARGET STOCK (by RMSE)
  Target TLKM: Trained on BBCA+ASII+UNVR + BiGRU (RMSE=61.5297, R²=0.977839)
  Target BBCA: Trained on TLKM+ASII+UNVR + GRU (RMSE=119.2748, R²=0.986838)
  Target ASII: Trained on TLKM+BBCA+UNVR + LSTM (RMSE=96.4832, R²=0.974804)
  Target UNVR: Trained on TLKM+BBCA+ASII + GRU (RMSE=76.8261, R²=0.993215)
